# NIFTY 50 Stock Price Prediction & Quantitative Machine Learning Pipeline
**Platform**: NIFTY-Pulse AI  
**Models**: PyTorch LSTM Deep Learning, Random Forest Regressor, Time-Series Prophet, Monte Carlo Simulations  
---
This notebook demonstrates an end-to-end quantitative financial engineering pipeline for predicting NIFTY 50 (`^NSEI`) and constituent stock prices (e.g., TCS, Reliance, Infosys).

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

## 1. Live Data Ingestion via Yahoo Finance API
We download historical daily OHLCV data for TCS (`TCS.NS`) or NIFTY 50 index.

In [ ]:
ticker = 'TCS.NS'
df = yf.download(ticker, period='5y', interval='1d')
print(f'Data shape: {df.shape}')
df.tail()

## 2. Technical Feature Engineering (RSI, MACD, Moving Averages)

In [ ]:
# Calculate 14-period RSI
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / (loss + 1e-9)
df['RSI'] = 100 - (100 / (1 + rs))

# Moving Averages
df['SMA_20'] = df['Close'].rolling(20).mean()
df['SMA_50'] = df['Close'].rolling(50).mean()

# MACD
ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema12 - ema26
df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()

df = df.dropna()
print('Feature engineering complete.')

## 3. Deep Learning PyTorch LSTM Model Implementation

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, output_dim=1):
        super(StockLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

print('LSTM Architecture Defined.')

## 4. Model Training & Evaluation

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
close_data = scaler.fit_transform(df[['Close']].values)

lookback = 60
X, y = [], []
for i in range(lookback, len(close_data)):
    X.append(close_data[i-lookback:i, 0])
    y.append(close_data[i, 0])
X, y = np.array(X), np.array(y)

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)

model = StockLSTM(input_dim=1, hidden_dim=64, num_layers=2, output_dim=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

epochs = 30
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = criterion(output, y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds_scaled = model(X_test_t).numpy()
preds_test = scaler.inverse_transform(preds_scaled)
y_test_orig = scaler.inverse_transform(y_test.reshape(-1, 1))

rmse = np.sqrt(mean_squared_error(y_test_orig, preds_test))
mae = mean_absolute_error(y_test_orig, preds_test)
r2 = r2_score(y_test_orig, preds_test)
print(f'Model Metrics: RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.4f}')